In [ ]:
import pandas as pd
import numpy as np
import umap
import joblib
from pathlib import Path
import os
import glob


In [ ]:

#if you have a lot of data for umaps
"""
file_location = os.path.join('C:\\Users\Austin\Documents\\UMAP_projects\\nsclc_model\\22_02_14_runs\\', '*.txt')
filenames = glob.glob(file_location)
"""

#Important params to set the window size and step time distance
init_dt = 0.01                #initial dt from the raw model output
dt = 0.1                      #dt for umap window
window = 4                    #window size in hours
time_step = 0.2                 #time step between windows

#umap params
neighbs = 50
min_d = 0.05
reducer = joblib.load(str(SAVEDIR) + f'\\reducer_04_10_{init_dt}_{dt}_{window}_{time_step}.sav')

In [ ]:
#Load and resample time based on umap window
time_tmp = pd.read_csv('t_4-9_3_3.txt', sep=",", header=None)
time_tmp = time_tmp.T
time_tmp.columns = ["time"]
time_tmp2 = time_tmp.iloc[::int(dt//init_dt)].reset_index().drop(columns=['index'])

In [ ]:
#create window and total protein arrays
win_len = int(window/dt)
range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))


for f in filenames:
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,18))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    prot_tmp2 = prot_tmp.iloc[::int(dt//init_dt),:].reset_index().drop(columns=['index'])
    
    prot_re_tmp = np.empty((len(range_ind),win_len*18))
    time_re_tmp = np.empty((len(range_ind),win_len))
    time_at_tmp = np.empty((len(range_ind),1))

    ii=0
    for n in range(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt)):
        tmp = prot_tmp2.iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp=tmp.to_numpy()
        tmp=np.reshape(np.transpose(tmp),win_len*18)
        tmp_at = time_tmp2['time'].iloc[n]
        
        prot_re_tmp[ii,:] = np.array([tmp])
        time_re_tmp[ii,:] = np.array([tmp_t])
        time_at_tmp[ii,:] = np.array([[tmp_at]])
        ii+=1
    
    embedding_all_prot = reducer.transform(prot_re_tmp)
    split_s = f.split('.')
    split_ss = split_s[0].split('\\')
    name = split_ss[-1]+'.csv'
    
    np.savetxt(Path(SAVEDIR), f'embedding_{name}'), embedding_all_prot, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_time_{name}'), time_re_tmp, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_time_{name}'), time_at_tmp, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_prot_{name}'), prot_re_tmp, delimiter=',', fmt='%f')

In [ ]:
#create window and total protein arrays using max vals
win_len = int(window/dt)
range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))


max_prots=[0]*18
min_prots=[1000]*18

for f in filenames:
    tmp_max=[]
    tmp_min=[]
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,18))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    prot_tmp[prot_tmp<0] = 0

    tmp_max = [prot_tmp[col].max() for col in prot_tmp]
    tmp_min = [prot_tmp[col].min() for col in prot_tmp]

    max_prots = [max(max_prots[n], tmp_max[n]) for n in range(18)]
    min_prots = [min(min_prots[n], tmp_min[n]) for n in range(18)]

for f in filenames:
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,18))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    for ii in range(18):
        prot_tmp.iloc[:,ii] = (prot_tmp.iloc[:,ii] - min_prots[ii])/(max_prots[ii] - min_prots[ii])
    prot_tmp2 = prot_tmp.iloc[::int(dt//init_dt),:].reset_index().drop(columns=['index'])
    
    prot_re_tmp = np.empty((len(range_ind),win_len*18))
    time_re_tmp = np.empty((len(range_ind),win_len))
    time_at_tmp = np.empty((len(range_ind),1))

    ii=0
    for n in range(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt)):
        tmp = prot_tmp2.iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp=tmp.to_numpy()
        tmp=np.reshape(np.transpose(tmp),win_len*18)
        tmp_at = time_tmp2['time'].iloc[n]
        
        prot_re_tmp[ii,:] = np.array([tmp])
        time_re_tmp[ii,:] = np.array([tmp_t])
        time_at_tmp[ii,:] = np.array([[tmp_at]])
        ii+=1
    
    embedding_all_prot = reducer.transform(prot_re_tmp)
    split_s = f.split('.')
    name = 'embedding_erl_'+split_s[0]+'.csv'
 
    np.savetxt(Path(SAVEDIR), f'embedding_{split_s[0]}.csv'), embedding_all_prot, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_time_{split_s[0]}.csv'), time_re_tmp, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_time_{split_s[0]}.csv'), time_at_tmp, delimiter=',', fmt='%f')
    np.savetxt(Path(SAVEDIR), f'all_prot_{split_s[0]}.csv'), prot_re_tmp, delimiter=',', fmt='%f')